In [66]:
import pandas as pd
import numpy as np
from scipy.stats import randint
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

In [3]:
df = pd.read_csv("../data/raw/car-details.csv")

In [4]:
df.head()

,name,company,model,edition,year,owner,fuel,seller_type,transmission,km_driven,mileage_mpg,engine_cc,max_power_bhp,torque_nm,seats,selling_price
0,Maruti Swift Dzire VDI,Maruti,Swift,Dzire VDI,2014,First,Diesel,Individual,Manual,145500,55.00,1248.0,74.00,190.000000,5.0,450000
1,Skoda Rapid 1.5 TDI Ambition,Skoda,Rapid,1.5 TDI Ambition,2014,Second,Diesel,Individual,Manual,120000,49.70,1498.0,103.52,250.000000,5.0,370000
2,Honda City 2017-2020 EXi,Honda,City,2017-2020 EXi,2006,Third,Petrol,Individual,Manual,140000,41.60,1497.0,78.00,124.544455,5.0,158000
3,Hyundai i20 Sportz Diesel,Hyundai,i20,Sportz Diesel,2010,First,Diesel,Individual,Manual,127000,54.06,1396.0,90.00,219.668960,5.0,225000
4,Maruti Swift VXI BSIII,Maruti,Swift,VXI BSIII,2007,First,Petrol,Individual,Manual,120000,37.84,1298.0,88.20,112.776475,5.0,130000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6926 entries, 0 to 6925
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           6926 non-null   str    
 1   company        6926 non-null   str    
 2   model          6926 non-null   str    
 3   edition        6926 non-null   str    
 4   year           6926 non-null   int64  
 5   owner          6926 non-null   str    
 6   fuel           6926 non-null   str    
 7   seller_type    6926 non-null   str    
 8   transmission   6926 non-null   str    
 9   km_driven      6926 non-null   int64  
 10  mileage_mpg    6718 non-null   float64
 11  engine_cc      6718 non-null   float64
 12  max_power_bhp  6717 non-null   float64
 13  torque_nm      6717 non-null   float64
 14  seats          6718 non-null   float64
 15  selling_price  6926 non-null   int64  
dtypes: float64(5), int64(3), str(8)
memory usage: 865.9 KB


In [6]:
df.isna().sum()

name               0
company            0
model              0
edition            0
year               0
owner              0
fuel               0
seller_type        0
transmission       0
km_driven          0
mileage_mpg      208
engine_cc        208
max_power_bhp    209
torque_nm        209
seats            208
selling_price      0
dtype: int64

In [7]:
df.shape

(6926, 16)

In [8]:
for col in df.select_dtypes(include = 'str').columns:
    print(f"Column : {col}")
    print(f"Cardinality : {df[col].nunique()}")
    print(df[col].unique())
    print(df[col].value_counts(normalize = True))
    print()

Column : name
Cardinality : 2058
<StringArray>
[                      'Maruti Swift Dzire VDI',
                 'Skoda Rapid 1.5 TDI Ambition',
                     'Honda City 2017-2020 EXi',
                    'Hyundai i20 Sportz Diesel',
                       'Maruti Swift VXI BSIII',
                'Hyundai Xcent 1.2 VTVT E Plus',
                 'Maruti Wagon R LXI DUO BSIII',
                           'Maruti 800 DX BSII',
                             'Toyota Etios VXD',
         'Ford Figo Diesel Celebration Edition',
 ...
                    'Toyota Innova 2.5 E 7 STR',
                            'Honda City ZXi AT',
                 'Ford Fiesta 1.4 Duratorq EXI',
                           'Chevrolet Cruze LT',
                  'Maruti Celerio ZXI AMT BSIV',
                        'Tata Bolt Revotron XM',
           'Tata Manza Aura (ABS) Safire BS IV',
                   'Tata Nexon 1.5 Revotorq XT',
     'Ford Freestyle Titanium Plus Diesel BSIV',
 'Toyota Innova 2

In [9]:
df = df.drop(columns = ['model', "name", "edition"])

In [10]:
df.head()

,company,year,owner,fuel,seller_type,transmission,km_driven,mileage_mpg,engine_cc,max_power_bhp,torque_nm,seats,selling_price
0,Maruti,2014,First,Diesel,Individual,Manual,145500,55.00,1248.0,74.00,190.000000,5.0,450000
1,Skoda,2014,Second,Diesel,Individual,Manual,120000,49.70,1498.0,103.52,250.000000,5.0,370000
2,Honda,2006,Third,Petrol,Individual,Manual,140000,41.60,1497.0,78.00,124.544455,5.0,158000
3,Hyundai,2010,First,Diesel,Individual,Manual,127000,54.06,1396.0,90.00,219.668960,5.0,225000
4,Maruti,2007,First,Petrol,Individual,Manual,120000,37.84,1298.0,88.20,112.776475,5.0,130000


In [11]:
df.duplicated().sum()

np.int64(19)

In [12]:
df = df.drop_duplicates()

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
X = df.drop(columns = 'selling_price')
y = df.selling_price.copy()

In [15]:
X.shape, y.shape

((6907, 12), (6907,))

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 42)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(6216, 12) (6216,)
(691, 12) (691,)


In [18]:
num_cols = X_train.select_dtypes(include = 'number').columns.tolist()
cat_cols = [col for col in X_train.columns if col not in num_cols]

print(num_cols)
print(cat_cols)

['year', 'km_driven', 'mileage_mpg', 'engine_cc', 'max_power_bhp', 'torque_nm', 'seats']
['company', 'owner', 'fuel', 'seller_type', 'transmission']


In [67]:
num_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'missing')),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

preprocessor = ColumnTransformer(transformers = [
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])


rf_regressor = RandomForestRegressor(random_state = 42)

rf_pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('reg', rf_regressor)
])

# 2. Define Hyperparameters
# Focus on depth control and leaf size to fix the 40k gap
RF_PARAMS = {
    'reg__n_estimators': randint(100, 500),
    'reg__max_depth': randint(8, 20),
    'reg__min_samples_split': randint(2, 10),
    'reg__min_samples_leaf': randint(4, 20),  # Higher values prevent overfitting
    'reg__max_features': ['sqrt', 'log2', 0.3] # Limits features per split
}

# 3. Setup Randomized Search
rf_search = RandomizedSearchCV(
    estimator = rf_pipeline,
    param_distributions = RF_PARAMS,
    n_iter = 15,
    cv = 5,
    n_jobs = -1,
    verbose = 2,
    scoring = 'neg_root_mean_squared_error',
    random_state = 42
)

# 4. Fit the model
rf_search.fit(X_train, y_train)

# 5. Extract Best Model
best_rf_model = rf_search.best_estimator_

Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [68]:
y_pred_rf = best_rf_model.predict(X_test)
y_train_rf = best_rf_model.predict(X_train)

rf_test_rmse = root_mean_squared_error(y_test, y_pred_rf)
rf_train_rmse = root_mean_squared_error(y_train, y_train_rf)

print(f"RF Best Params: {rf_search.best_params_}")
print(f"RF Train RMSE: {rf_train_rmse:,.3f}")
print(f"RF Test RMSE: {rf_test_rmse:,.3f}")
print(f"Generalization Gap: {abs(rf_test_rmse - rf_train_rmse):,.3f}")

RF Best Params: {'reg__max_depth': 10, 'reg__max_features': 0.3, 'reg__min_samples_leaf': 7, 'reg__min_samples_split': 9, 'reg__n_estimators': 230}
RF Train RMSE: 172,110.286
RF Test RMSE: 182,361.571
Generalization Gap: 10,251.286


In [29]:
from lightgbm import LGBMRegressor

In [58]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

from scipy.stats import randint, uniform

# Focus on regularization to close the 51k RMSE gap
LIGHTGBM_PARAMS = {
    'n_estimators': randint(500, 1500),         
    'max_depth': randint(5, 15),                
    'learning_rate': uniform(0.01, 0.05),       
    'num_leaves': randint(20, 50),              
    'min_child_samples': randint(50, 200),      
    'reg_alpha': uniform(0, 10),                
    'reg_lambda': uniform(0, 10),               
    'boosting_type': ['gbdt', 'dart']           
}

RANDOM_SEARCH_PARAMS = {
    'n_iter': 15,                               
    'cv': 5,                                    
    'n_jobs': -1,
    'verbose': 2,
    'random_state': 42,
    'scoring': 'neg_root_mean_squared_error'     
}

In [59]:
lgbm_model =  LGBMRegressor(random_state = RANDOM_SEARCH_PARAMS["random_state"])

In [61]:
from sklearn.pipeline import Pipeline

# Create the LightGBM Pipeline
lgbm_pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('reg', lgbm_model)
])

# Update the Param Keys to match the pipeline structure (prefix with 'reg__')
LGBM_PIPELINE_PARAMS = {f'reg__{k}': v for k, v in LIGHTGBM_PARAMS.items()}

random_search = RandomizedSearchCV(
    estimator = lgbm_pipeline, # Use the pipeline here
    param_distributions = LGBM_PIPELINE_PARAMS,
    n_iter = RANDOM_SEARCH_PARAMS["n_iter"],
    cv = RANDOM_SEARCH_PARAMS["cv"],
    n_jobs = RANDOM_SEARCH_PARAMS["n_jobs"],
    verbose = RANDOM_SEARCH_PARAMS["verbose"],
    random_state = RANDOM_SEARCH_PARAMS["random_state"],
    scoring = RANDOM_SEARCH_PARAMS["scoring"]
)

random_search.fit(X_train, y_train)

# Extract the best details
best_params = random_search.best_params_
best_lgbm_model = random_search.best_estimator_
best_score = abs(random_search.best_score_)

print("-" * 30)
print("HYPERPARAMETER TUNING RESULTS")
print("-" * 30)
print(f"Best CV RMSE: {best_score:,.3f}")
print("\nBest Parameters Found:")
for param, value in best_params.items():
    # Clean up the prefix for readability
    clean_param = param.replace('reg__', '')
    print(f"  - {clean_param}: {value}")
print("-" * 30)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
------------------------------
HYPERPARAMETER TUNING RESULTS
------------------------------
Best CV RMSE: 175,214.438

Best Parameters Found:
  - boosting_type: gbdt
  - learning_rate: 0.04982714934301165
  - max_depth: 12
  - min_child_samples: 70
  - n_estimators: 1114
  - num_leaves: 45
  - reg_alpha: 1.5599452033620265
  - reg_lambda: 0.5808361216819946
------------------------------


In [62]:
# 1. Predict on the test set
y_pred = best_lgbm_model.predict(X_test)

# 2. Calculate Final Test RMSE
final_test_rmse = root_mean_squared_error(y_test, y_pred)

# 3. Calculate Training RMSE to check for the final gap
y_train_pred = best_lgbm_model.predict(X_train)
final_train_rmse = root_mean_squared_error(y_train, y_train_pred)

print(f"Final Train RMSE: {final_train_rmse:,.3f}")
print(f"Final Test RMSE: {final_test_rmse:,.3f}")
print(f"Generalization Gap: {final_test_rmse - final_train_rmse:,.3f}")

Final Train RMSE: 122,471.755
Final Test RMSE: 154,758.640
Generalization Gap: 32,286.885


c:\Users\sumit\OneDrive\Desktop\Code_PlayGround\End To End ML Projects\Car_Price_Prediction_API\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\sumit\OneDrive\Desktop\Code_PlayGround\End To End ML Projects\Car_Price_Prediction_API\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [63]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# 1. Define the Pipeline
ridge_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('reg', Ridge())
])

# 2. Define Hyperparameters
# 'alpha' is the regularization strength (higher = more constraint)
RIDGE_PARAMS = {
    'reg__alpha': uniform(0.1, 100.0), 
    'reg__solver': ['svd', 'cholesky', 'lsqr', 'sag']
}

# 3. Setup Randomized Search
ridge_search = RandomizedSearchCV(
    estimator = ridge_model,
    param_distributions = RIDGE_PARAMS,
    n_iter = 20,
    cv = 5,
    n_jobs = -1,
    scoring = 'neg_root_mean_squared_error',
    random_state = 42
)

# 4. Fit and Extract
ridge_search.fit(X_train, y_train)

best_ridge_model = ridge_search.best_estimator_
print(f"Best Ridge Alpha: {ridge_search.best_params_['reg__alpha']:.4f}")

Best Ridge Alpha: 0.1779


c:\Users\sumit\OneDrive\Desktop\Code_PlayGround\End To End ML Projects\Car_Price_Prediction_API\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [65]:
y_pred_ridge = best_ridge_model.predict(X_test)
y_train_ridge = best_ridge_model.predict(X_train)

# Metrics
ridge_test_rmse = root_mean_squared_error(y_test, y_pred_ridge)
ridge_train_rmse = root_mean_squared_error(y_train, y_train_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print(f"Ridge Train RMSE: {ridge_train_rmse:,.3f}")
print(f"Ridge Test RMSE: {ridge_test_rmse:,.3f}")
print(f"Generalization Gap: {abs(ridge_test_rmse - ridge_train_rmse):,.3f}")
print(f"Ridge R2 Score: {ridge_r2:.4f}")

Ridge Train RMSE: 257,364.579
Ridge Test RMSE: 290,023.081
Generalization Gap: 32,658.502
Ridge R2 Score: 0.6444


In [69]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error, r2_score

# 1. Define the Pipeline
lr_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('reg', LinearRegression())
])

# 2. Fit the model
lr_model.fit(X_train, y_train)

# 3. Perform 5-fold Cross-Validation (to compare with your LightGBM CV score)
# Note: neg_root_mean_squared_error returns negative values, so we take the absolute
cv_scores = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
cv_rmse = abs(cv_scores.mean())

print(f"Linear Regression CV RMSE: {cv_rmse:,.3f}")

Linear Regression CV RMSE: 273,935.897
